# nts-lookup-v3: continuing from v2 (Qwen3.5-4B + LoRA)

**What changes from the v2 notebook below (everything else is the same, proven set-up):**

| | v2 | **v3** |
|---|---|---|
| Answer | one lookup: `{"tool": ..., ...}` | **every lookup the message asks for**: `{"lookups": [{"tool": ...}, ...]}` (1–5) |
| Tools | 6 lookups | + calendar, my_requests, my_rules, clash_check, move_options, counts, gaps, my_fairness, bookings, what_if; `room_for`, several `weeks` |
| Prompt | `SYSTEM` + the department's lists (~1,650 tokens a row) | `SYSTEM_V3`: compact, **no lists** of teachers, sections, rooms or courses (names are written as the message writes them; the code finds who or what they are), ~1,300 tokens a row |
| Start | a fresh adapter | **nts-lookup-v2's adapter**, trained further (lower learning rate, 5e-5), with v2's kinds of question replayed so nothing is forgotten |
| A run cut short | lost | **resumes** from its last checkpoint in a new session |

**Inputs to add (Add Input):**
1. `data/lookup-train-v3` (made by `python -m training.lookup_data_v3`, plus Gemini's rewordings from
   `python -m training.lookup_distill_v3`, `distilled.jsonl` in the same folder, used when present);
2. nts-lookup-v2's **`lora/` folder** (the output of v2's run: `adapter_config.json` + `adapter_model.safetensors`).
   Without it, v3 trains a fresh adapter (then use `LR = 2e-4` and all the rows);
3. only to resume: the earlier session's **`checkpoints/`** folder (save the earlier session's output as a dataset).

First `TRIAL = True` (a few minutes: installs, loads, 20 steps, exports), then `TRIAL = False` and *Run all*.

---


# Training the lookup adapter (Qwen3.5-4B + LoRA)

This notebook teaches a small model to do what Gemini does in `agents/tool_agent.py`: read a question about
the timetable and answer with **which lookup to do**, as a small JSON. The code then runs the lookup and writes
the reply, so the model never states a fact itself.

```
question  ->  model  ->  {"tool": "free_hours", "teachers": ["Dr. Iyer"], "days": ["Fri"], "week": 13}
```

**What comes out:** a LoRA adapter (`nts-lookup-v1.gguf`, tens of MB). It goes on the same base model the team's
server already runs (`Qwen3.5-4B`), next to `nts-intents-v6-read100` and `nts-read100`. Nothing else on the
server changes.

### Before you run it
1. **Make the data** on your machine: `uv run python -m training.lookup_data`
   (writes `data/lookup-train/train.jsonl`, `val.jsonl`, `manifest.json`).
2. **Upload** the folder `data/lookup-train` to Kaggle as a dataset, and add it to this notebook (*Add Input*).
3. **Settings:** Accelerator **GPU T4 x2**, Internet **on**.
4. First run with `TRIAL = True` (20 steps, a few minutes) to check that everything installs and fits.
   Then set `TRIAL = False` and *Run all*.

### The steps
1. Install the libraries
2. Settings
3. Load the data and **check every row**
4. Load the model and add the LoRA adapter
5. Train
6. Quick check on the validation questions
7. Save the adapter and convert it to GGUF

## 1. Install the libraries

In [ ]:
%%capture
# True: also install flash-linear-attention, the fast kernels for Qwen3.5's linear-attention layers. Without it
# transformers says "falling back to its reference PyTorch implementation ... much slower". It may not support
# the T4: if this cell or training fails with it, set it back to False.
FAST_KERNELS = False
# Keep Kaggle's own CUDA build of torch: no other package may replace it.
!pip freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/torch-pin.txt
# transformers v5 is needed for Qwen3.5; PEFT adds the LoRA adapter; TRL has the training loop (SFTTrainer).
!pip install --upgrade -c /tmp/torch-pin.txt "transformers>=5" peft trl accelerate datasets
# Kaggle ships an old torchao that PEFT refuses; nothing here uses it.
!pip uninstall -y -q torchao
if FAST_KERNELS:
    !pip install -c /tmp/torch-pin.txt flash-linear-attention

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # fewer out-of-memory errors from fragmentation

import torch, transformers, peft, trl

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, "Qwen3.5 needs transformers v5: restart the kernel and run from here"

N_GPU = torch.cuda.device_count()
assert N_GPU > 0, "no GPU: set Settings > Accelerator > GPU T4 x2"
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")

## 2. Settings

Everything you might change is here.

In [ ]:
import glob, json, random, time
from pathlib import Path

SESSION_START = time.time()  # Kaggle stops a session after 12 hours, counted from here

MODEL = "Qwen/Qwen3.5-4B"   # the SAME base the team's adapters use: an adapter only works on its own base
ADAPTER = "nts-lookup-v3"   # the adapter's name: the GGUF file and its name in adapters-lora.toml

TRIAL = False               # True: 20 steps only, to check the setup (and still export, to test the server)
PRECISION = "fp32"          # "fp32": slower but safe. "fp16": faster, but Qwen3.5's gradients overflowed in
                            # fp16 here (grad_norm nan and learning_rate 0 on every step: nothing was learned)
EPOCHS = 1                  # passes over the training rows (one is a lot for this task; raise it if time allows)
MAX_TRAIN_ROWS = None       # e.g. 800: train on a random part of the rows, for a shorter run (None: all)
TIME_BUDGET_H = 11.0        # stop training after this many hours of the session, saving the adapter, so the
                            # export below still runs inside Kaggle's 12 hours
SAVE_EVERY = 25             # also save a checkpoint every this many steps
LR = 5e-5                   # continuing from v2: a lower rate adjusts what it knows instead of overwriting it
MAX_LEN = 2048              # tokens per row; v3's rows are about 1,300 (checked below)
BATCH = 1                   # rows per step on the GPU (1 keeps memory low on a T4)
ACCUM = 16                  # steps added up before each update: an effective batch of 16 rows
LORA_R = 16                 # the adapter's size (rank); 16 is the team's choice
CHECK_ROWS = 80             # validation questions to try at the end (generation is slow)
SEED = 3407

# The data: the dataset added to this notebook, wherever Kaggle mounted it.
def _version(train_file):
    m = Path(train_file).parent / "manifest.json"
    return json.loads(m.read_text()).get("version", 2) if m.exists() else 2


found = [f for f in sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)) if _version(f) == 3]
assert found, "no v3 train.jsonl: add the data/lookup-train-v3 folder as a dataset (Add Input)"
DATA = Path(found[0]).parent

# The adapter to continue from: nts-lookup-v2's lora/ folder, added as an input (None: a fresh adapter).
starts = [p for p in sorted(glob.glob("/kaggle/input/**/lora/adapter_config.json", recursive=True))
          if ADAPTER not in p]  # another adapter's final lora/ (v2's), not this one's
START_ADAPTER = Path(starts[0]).parent if starts else None
# A run of THIS adapter cut short in an earlier session: its checkpoints, added as an input (None: step 0).
# Only this adapter's (its name is in the path): v2's checkpoints, in the same input as v2's lora/, are not v3's.
ckpts = sorted((p for p in glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
                if ADAPTER in p), key=lambda p: int(Path(p).parent.name.split("-")[1]))
RESUME = str(Path(ckpts[-1]).parent) if ckpts else None
print("continue from:", START_ADAPTER or "a fresh adapter", "| resume:", RESUME or "no")
OUT = Path("/kaggle/working") / ADAPTER
OUT.mkdir(parents=True, exist_ok=True)
print("data:", DATA, "| output:", OUT)
random.seed(SEED)
torch.manual_seed(SEED)

## 3. Load the data and check every row

Each row is one chat, exactly as the app sends it to the model:

| message | content |
|---|---|
| `system` | the tools and the rules (`SYSTEM` in `agents/tool_agent.py`) |
| `user` | who asks, today and the time, the week, the days and hours, the department's names, and the question |
| `assistant` | **the answer to learn**: the lookup as JSON |

The data was already checked when it was made (each answer went through `ToolAgent`'s own checks). Here it is
checked again, so a broken upload or a hand-edited row cannot slip into training:
* three messages, in the order system, user, assistant;
* the answer is valid JSON, has a known `tool` and a `week`, and only fields the app knows;
* names, days and hours have the right types.

Rows that fail are listed and left out.

In [ ]:
from collections import Counter

TOOLS = {"free_hours", "classes_of", "free_rooms", "who_is_free", "describe", "changes", "calendar", "my_requests",
         "my_rules", "clash_check", "move_options", "counts", "gaps", "my_fairness", "bookings", "what_if",
         "unanswerable", "none"}
LIST_FIELDS = {"teachers", "sections", "rooms", "days", "dates", "hours", "equipment", "unknown", "ambiguous",
               "roles", "room_for", "weeks"}  # roles: v2; room_for, weeks: v3
OTHER_FIELDS = {"tool", "reason", "course", "part", "length", "kind", "min_capacity", "next", "count", "of",
                "what", "week", "first", "change",  # first, change: v2
                "which", "request_id", "measure", "order", "class_day", "class_kind", "club", "request",  # v3
                "separately"}  # v3: each one on its own, not together
DAYS = {"Mon", "Tue", "Wed", "Thu", "Fri"}


def problem(row):
    # Why a row must not be trained on, or None when it is fine.
    msgs = row.get("messages")
    if not isinstance(msgs, list) or [m.get("role") for m in msgs] != ["system", "user", "assistant"]:
        return "messages are not system, user, assistant"
    if "<question>" not in msgs[1]["content"]:
        return "the user message has no <question>"
    try:
        lookups = json.loads(msgs[2]["content"]).get("lookups")
    except (json.JSONDecodeError, AttributeError):
        return "the answer is not a JSON object"
    if not isinstance(lookups, list) or not 1 <= len(lookups) <= 5:
        return "the answer has no list of 1 to 5 lookups"
    for answer in lookups:
        if why := problem_of(answer):
            return why
    return None


def problem_of(answer):
    # one lookup in the list
    if not isinstance(answer, dict) or answer.get("tool") not in TOOLS:
        return f"unknown tool {answer.get('tool')!r}"
    if not isinstance(answer.get("week"), int):
        return "no week number"
    if extra := set(answer) - LIST_FIELDS - OTHER_FIELDS:
        return f"unknown fields {sorted(extra)}"
    if bad := [k for k in LIST_FIELDS & set(answer) if not isinstance(answer[k], list)]:
        return f"not lists: {bad}"
    if bad := [d for d in answer.get("days", []) if d not in DAYS]:
        return f"days {bad}"
    if bad := [h for h in answer.get("hours", []) if not isinstance(h, int) or not 0 <= h <= 7]:
        return f"hours {bad}"
    return None


def load(name):
    rows, dropped = [], []
    for n, line in enumerate(open(DATA / f"{name}.jsonl", encoding="utf-8"), 1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            dropped.append((n, "not a JSON line"))
            continue
        why = problem(row)
        if why:
            dropped.append((n, why))
        else:
            rows.append(row)
    print(f"{name}: {len(rows)} rows kept, {len(dropped)} left out")
    for n, why in dropped[:20]:
        print(f"   line {n}: {why}")
    return rows


train_rows, val_rows = load("train"), load("val")
if (DATA / "distilled.jsonl").exists():  # Gemini's rewordings (training.lookup_distill_v3): training rows too
    train_rows += load("distilled")
assert train_rows and val_rows, "no usable rows"
if MAX_TRAIN_ROWS and len(train_rows) > MAX_TRAIN_ROWS:  # a shorter run: a random part, every kind still in it
    train_rows = random.Random(SEED).sample(train_rows, MAX_TRAIN_ROWS)
    print(f"training on {len(train_rows)} of the rows (MAX_TRAIN_ROWS)")

# What the data covers: every tool, every kind of question, both kinds of asker, several departments.
def tools_of(row):
    return [x["tool"] for x in json.loads(row["messages"][2]["content"])["lookups"]]

print("\ntools:      ", dict(Counter(t for r in train_rows for t in tools_of(r)).most_common()))
print("lookups per message:", dict(Counter(len(tools_of(r)) for r in train_rows).most_common()))
print("kinds:      ", dict(Counter(r.get("kind") for r in train_rows).most_common()))
print("groups:     ", dict(Counter(r.get("group") for r in train_rows).most_common()))
print("departments:", dict(Counter(r.get("department") for r in train_rows).most_common()))
fields = Counter(k for r in train_rows for x in json.loads(r["messages"][2]["content"])["lookups"] for k in x)
print("fields:     ", dict(fields.most_common()))
missing = TOOLS - {t for r in train_rows for t in tools_of(r)} - {"unanswerable"}
assert not missing, f"no training rows for {missing}: make the data again"

In [ ]:
# One row, as the model will see it.
example = train_rows[0]["messages"]
print(example[1]["content"][-700:])
print("\nANSWER TO LEARN:", example[2]["content"])

## 4. Load the model and add the LoRA adapter

* **The base model** is loaded in `PRECISION` (fp32: about 17 GB) and split over the two T4s
  (`device_map="auto"`). Its own weights stay frozen.
* **The LoRA adapter** adds small trainable matrices to the attention and MLP layers of the *language*
  part only (Qwen3.5 also has a vision part, which we don't use). That is about 1% of the weights, and the
  only part that is trained.
* The adapter's own weights are always float32.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.chat_template is None:  # some releases keep the chat template with the processor
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODEL).chat_template

DTYPE = torch.float32 if PRECISION == "fp32" else torch.float16
model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype=DTYPE, device_map="auto", attn_implementation="sdpa")
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GB of weights")
model.config.use_cache = False  # not needed while training; it would only use memory
print("placed on:", sorted({str(d) for d in model.hf_device_map.values()}))

if START_ADAPTER:  # v2's adapter, trained further: it starts knowing everything v2 knows
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(START_ADAPTER), is_trainable=True)
    print("continuing from", START_ADAPTER)
else:
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_R, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
        # every attention and MLP projection, but not in the vision part ("visual")
        target_modules=r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"))
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()  # the adapter in float32 (with fp16, the frozen base stays fp16)
model.print_trainable_parameters()

## 5. Train

Each row becomes a **prompt** (system + user, rendered with the chat template, thinking off, as the server
does) and a **completion** (the JSON answer + end of text). The loss is computed on the completion only: the
model learns to *answer*, not to repeat the prompt.

**Progress:** after every step a line with the loss, the seconds per step and the estimated finish, so you know
within minutes whether the run fits in Kaggle's 12 hours. A checkpoint is saved every `SAVE_EVERY` steps, and
training stops by itself at `TIME_BUDGET_H` with the adapter as far as it got, so a long run is never lost.

In [ ]:
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer


class Clock(TrainerCallback):
    # One line per step: the loss, seconds per step, and when training will finish. Stops at TIME_BUDGET_H,
    # and stops a run that learns nothing (the gradients not a number on 8 steps in a row: fp16 overflow).
    def on_train_begin(self, args, state, control, **kwargs):
        self.start, self.nan_steps = time.time(), 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return
        grad = float(logs.get("grad_norm", 0.0))
        self.nan_steps = self.nan_steps + 1 if grad != grad or grad == float("inf") else 0
        if self.nan_steps >= 8:
            print("STOPPED: the gradients were not a number on 8 steps in a row, so nothing is learned. "
                  "Use PRECISION = 'fp32'.", flush=True)
            control.should_training_stop = True
        done, total = state.global_step, state.max_steps
        per_step = (time.time() - self.start) / max(done, 1)
        left_h = per_step * (total - done) / 3600
        print(f"step {done}/{total}  loss {logs['loss']:.4f}  {per_step:.0f} s/step  "
              f"about {left_h:.1f} h left", flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        if not TRIAL and (time.time() - SESSION_START) / 3600 > TIME_BUDGET_H:
            print(f"time budget reached at step {state.global_step}: stopping, saved as a checkpoint (the next session resumes from it) and as the adapter below", flush=True)
            control.should_save = True
            control.should_training_stop = True



def prompt_of(messages):
    # system + user, as the server renders them (thinking off), ending where the answer starts
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)


def to_example(row):
    return {"prompt": prompt_of(row["messages"]), "completion": row["messages"][2]["content"] + tok.eos_token}


train_ds = Dataset.from_list([to_example(r) for r in train_rows])
val_ds = Dataset.from_list([to_example(r) for r in val_rows])

lengths = [len(tok(e["prompt"] + e["completion"])["input_ids"]) for e in train_ds]
print(f"tokens per row: max {max(lengths)}, mean {sum(lengths) // len(lengths)}")
assert max(lengths) <= MAX_LEN, "a row is longer than MAX_LEN: raise MAX_LEN"

trainer = SFTTrainer(
    model=model,
    processing_class=tok,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    # the settings of the team's notebook (train_multigpu_kaggle), which train on two T4s with transformers v5
    args=SFTConfig(
        output_dir=str(OUT / "checkpoints"),
        max_length=MAX_LEN,
        completion_only_loss=True,          # learn the answer, not the prompt
        loss_type="nll",                     # TRL's default loss breaks when the model is split over two GPUs
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS,
        max_steps=20 if TRIAL else -1,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=5,                      # a gentle start (transformers v5 has no warmup_ratio any more)
        optim="adamw_torch",                 # only the adapter trains, so the optimizer's memory is small
        fp16=PRECISION == "fp16",            # mixed precision only with fp16 (see PRECISION)
        bf16=False,                          # T4s have no bf16
        gradient_checkpointing=True,         # less memory, a little slower
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1,                     # a progress line every step (see Clock)
        disable_tqdm=True,                   # the progress bar does not show in Kaggle's log; Clock does
        eval_strategy="no",                  # the quick check below does this, on unseen wordings
        save_strategy="steps",
        save_steps=SAVE_EVERY,
        save_total_limit=2,                  # the last two, so a session cut mid-save still has one
        report_to="none",
        seed=SEED,
    ),
    callbacks=[Clock()],
)
stats = trainer.train(resume_from_checkpoint=RESUME)  # RESUME: an earlier session's last checkpoint, or None
print(f"trained {trainer.state.global_step} steps in {stats.metrics['train_runtime'] / 60:.0f} min")

## 6. Quick check on the validation questions

The validation rows come from templates the model has **never seen** (wordings kept out of training), so this
shows whether it learned the task or only the training wordings. For each question: is the answer valid JSON,
and is it the same lookup as the expected one (empty fields ignored, lists in any order)?

This is only a sanity check. The real score comes from the project's test set, with the adapter served:
`uv run python -m evaluation.lookup_set` (see the end of this notebook).

In [ ]:
model.eval()
model.config.use_cache = True


def same(a, b):
    # the same lookup: empty fields don't count, lists in any order
    def norm(o):
        return {k: sorted(v) if isinstance(v, list) else v for k, v in o.items() if v not in (None, [], False)}
    return norm(a) == norm(b)


rows = random.Random(SEED).sample(val_rows, min(CHECK_ROWS, len(val_rows)))
score, misses = Counter(), []
for row in rows:
    inputs = tok(prompt_of(row["messages"]), return_tensors="pt").to("cuda:0")  # the first layers are on GPU 0
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=500, do_sample=False, use_cache=True)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    want = json.loads(row["messages"][2]["content"])
    score["questions"] += 1
    try:
        got = json.loads(text)
        score["valid JSON"] += 1
        ok = (isinstance(got, dict) and len(got.get("lookups", [])) == len(want["lookups"])
              and all(same(g, w) for g, w in zip(got["lookups"], want["lookups"])))
    except json.JSONDecodeError:
        ok = False
    score["right"] += ok
    for t in {x["tool"] for x in want["lookups"]}:
        score[f"right {t}"] += ok
        score[f"asked {t}"] += 1
    score[f"right {len(want['lookups'])} lookups"] += ok
    score[f"asked {len(want['lookups'])} lookups"] += 1
    if not ok:
        misses.append((row["messages"][1]["content"].split("<question>")[1].split("</question>")[0], want, text))

print(f"{score['right']}/{score['questions']} right, {score['valid JSON']} valid JSON")
for n in range(1, 6):
    if score[f"asked {n} lookups"]:
        print(f"   {n} lookup(s)   {score[f'right {n} lookups']}/{score[f'asked {n} lookups']}")
for t in sorted(TOOLS):
    if score[f"asked {t}"]:
        print(f"   {t:13} {score[f'right {t}']}/{score[f'asked {t}']}")
for question, want, text in misses[:15]:
    print(f"\nQ: {question}\n   want {json.dumps(want)}\n   got  {text}")

## 7. Save the adapter and convert it to GGUF

* `lora/`: the adapter as PEFT saves it (keep it: it can be trained further later);
* `gguf/nts-lookup-v1.gguf`: the same adapter for `llama-server`, made with llama.cpp's
  `convert_lora_to_gguf.py` **at the server's build (b11368)**, as the team's notebook does, because a newer
  converter can write a file an older server cannot read.

In [ ]:
model.save_pretrained(str(OUT / "lora"))
tok.save_pretrained(str(OUT / "lora"))
(OUT / "manifest.json").write_text(json.dumps({
    "adapter": ADAPTER, "base": MODEL, "task": "lookup", "train_rows": len(train_rows), "val_rows": len(val_rows),
    "epochs": EPOCHS, "lr": LR, "lora_r": LORA_R, "trial": TRIAL, "version": 3,
    "start_adapter": str(START_ADAPTER) if START_ADAPTER else None, "resumed_from": RESUME,
    "check": {k: v for k, v in score.items() if k in ("questions", "valid JSON", "right")}}, indent=1))
print("saved", OUT / "lora")

In [ ]:
from huggingface_hub import snapshot_download

!git clone --depth 1 --branch b11368 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q /tmp/llama.cpp/gguf-py sentencepiece
BASE_DIR = snapshot_download(MODEL)  # the base's config (already downloaded)
(OUT / "gguf").mkdir(exist_ok=True)
!python /tmp/llama.cpp/convert_lora_to_gguf.py {OUT}/lora --base {BASE_DIR} --outtype f16 --outfile {OUT}/gguf/{ADAPTER}.gguf
for f in sorted((OUT / "gguf").glob("*.gguf")):
    print(f, f"{f.stat().st_size / 2**20:.0f} MB")

## Next: serve it and measure it

1. Download `nts-lookup-v1/gguf/nts-lookup-v1.gguf` (and keep `lora/` and `manifest.json`) from the
   notebook's *Output* tab.
2. On the server, add it next to the others (same base, nothing else changes):
   ```sh
   llama-server -m Qwen3.5-4B-Base-Q8_0.gguf \
       --lora nts-intents-v6-read100.gguf,nts-read100.gguf,nts-lookup-v1.gguf \
       --lora-init-without-apply --jinja -ngl 99 -c 8192 --port 8080
   ```
3. In the project, `adapters-lora.toml` gets a line `nts-lookup-v1 = ["lookup"]`, and `lookup` must be added
   to the known tasks (`TASKS` in `src/language/adapters.py`). Then `ToolAgent` can use the local client
   instead of Gemini.
4. Measure it on the **test set** (templates it never saw), against the regex and Gemini:
   `uv run python -m evaluation.lookup_set ...` (Gemini 3.8-flash: 320/362; the regex: 242/362).